# Forest Fire AI — Final Inference Testing
## Notebook 10: End-to-End Testing of Image + Tabular Models

Tests all saved model artifacts exactly as the dashboard would use them.


In [ ]:
import os, sys, json, pickle, time, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image

ROOT     = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL     = ROOT / "Implementation"
MODEL_DIR_IMG = IMPL / "models" / "image"
MODEL_DIR_TAB = IMPL / "models" / "tabular"
PRED_DIR = IMPL / "artifacts" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Final Inference Testing")
print("="*60)
print(f"Device: {DEVICE}")


In [ ]:
# ── Load image model ──────────────────────────────────────────────────
with open(MODEL_DIR_IMG / "class_names.json") as f:
    cnames = json.load(f)
with open(MODEL_DIR_IMG / "model_metadata.json") as f:
    mmeta = json.load(f)

CLASS_NAMES  = cnames['class_names']
CLASS_TO_IDX = cnames['class_to_idx']
IDX_TO_CLASS = {int(k): v for k, v in cnames['idx_to_class'].items()}
MODEL_NAME   = mmeta['model_name']
IMAGE_SIZE   = mmeta['image_size']
IMG_MEAN     = mmeta['imagenet_mean']
IMG_STD      = mmeta['imagenet_std']

def build_model(model_name, num_classes=2):
    if model_name == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'MobileNetV3':
        m = models.mobilenet_v3_small(weights=None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif model_name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'ResNet18':
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

img_model = build_model(MODEL_NAME)
img_model.load_state_dict(torch.load(MODEL_DIR_IMG / "best_model.pth", map_location=DEVICE))
img_model = img_model.to(DEVICE)
img_model.eval()
print(f"✓ Image model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in img_model.parameters()):,}")

transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD)
])

def predict_image(img_path_or_pil, model=img_model):
    if isinstance(img_path_or_pil, (str, Path)):
        img = Image.open(img_path_or_pil).convert('RGB')
    else:
        img = img_path_or_pil.convert('RGB')
    inp = transform(img).unsqueeze(0).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out   = model(inp)
        probs = F.softmax(out, dim=1)[0]
    inf_ms = (time.time() - t0) * 1000
    pred_idx = probs.argmax().item()
    pred_cls = IDX_TO_CLASS[pred_idx]
    conf     = probs[pred_idx].item()
    class_probs = {CLASS_NAMES[i]: round(probs[i].item(), 4) for i in range(len(CLASS_NAMES))}
    return {'prediction': pred_cls, 'confidence': round(conf, 4),
            'class_probs': class_probs, 'inference_ms': round(inf_ms, 2)}


In [ ]:
# ── Test image model ──────────────────────────────────────────────────
from pathlib import Path
import json

with open(IMPL / "artifacts" / "metadata" / "data_splits.json") as f:
    splits = json.load(f)

test_data = splits['test']
fire_tests    = [(p, l) for p, l in test_data if l == 'FIRE'][:3]
nofire_tests  = [(p, l) for p, l in test_data if l == 'NO_FIRE'][:3]

print("Image Inference Tests:")
print("="*60)
for img_path, true_label in fire_tests + nofire_tests:
    try:
        result = predict_image(img_path)
        correct = "✓" if result['prediction'] == true_label else "✗"
        print(f"  {correct} True: {true_label:<8} | Pred: {result['prediction']:<8} | "
              f"Conf: {result['confidence']:.4f} | Inf: {result['inference_ms']:.1f}ms")
    except Exception as e:
        print(f"  Error: {e}")


In [ ]:
# ── Test different image sizes ────────────────────────────────────────
print("\nTest different input sizes:")
test_img_path = fire_tests[0][0] if fire_tests else None
if test_img_path:
    for size in [(64,64), (128,128), (224,224), (512,512), (1024,1024)]:
        img = Image.open(test_img_path).convert('RGB').resize(size)
        try:
            result = predict_image(img)
            print(f"  Size {size}: {result['prediction']} ({result['confidence']:.3f}) | {result['inference_ms']:.1f}ms")
        except Exception as e:
            print(f"  Size {size}: Error — {e}")


In [ ]:
# ── Load tabular models ───────────────────────────────────────────────
clf    = pickle.load(open(MODEL_DIR_TAB / "classifier.pkl", "rb"))
reg    = pickle.load(open(MODEL_DIR_TAB / "regressor.pkl",  "rb"))
scaler = pickle.load(open(MODEL_DIR_TAB / "scaler.pkl",     "rb"))
le_month = pickle.load(open(MODEL_DIR_TAB / "le_month.pkl", "rb"))
le_day   = pickle.load(open(MODEL_DIR_TAB / "le_day.pkl",   "rb"))
with open(MODEL_DIR_TAB / "metadata.json") as f:
    tmeta = json.load(f)

FEATURE_COLS = tmeta['feature_cols']
NUM_FEATS    = tmeta['numerical_features']

print("✓ Tabular models loaded")
print(f"  Classifier: {tmeta['classifier_name']}")
print(f"  Regressor:  {tmeta['regressor_name']}")
print(f"  Features:   {FEATURE_COLS}")

def predict_tabular(row_dict):
    """Predict fire risk from a dict of environmental features."""
    num_vals = [float(row_dict.get(f, 0)) for f in NUM_FEATS]
    month_enc = le_month.transform([row_dict.get('month', 'aug').lower()])[0]
    day_enc   = le_day.transform([row_dict.get('day', 'fri').lower()])[0]
    X = np.array(num_vals + [month_enc, day_enc]).reshape(1, -1)
    fire_prob  = clf.predict_proba(X)[0, 1]
    fire_pred  = int(clf.predict(X)[0])
    area_log   = reg.predict(X)[0]
    area_pred  = float(np.expm1(area_log))
    return {
        'fire_predicted': fire_pred,
        'fire_probability': round(float(fire_prob), 4),
        'predicted_area_ha': round(area_pred, 4)
    }


In [ ]:
# Test tabular model
print("\nTabular Inference Tests:")
print("="*60)
test_rows = [
    {'X':7,'Y':5,'month':'aug','day':'fri','FFMC':92.3,'DMC':85.3,'DC':488.0,'ISI':14.7,'temp':22.2,'RH':29,'wind':5.4,'rain':0.0},
    {'X':4,'Y':4,'month':'feb','day':'mon','FFMC':73.2,'DMC':14.0,'DC':25.6,'ISI':2.0, 'temp':4.5,'RH':80,'wind':1.3,'rain':0.0},
    {'X':6,'Y':5,'month':'sep','day':'sat','FFMC':93.4,'DMC':145.4,'DC':721.4,'ISI':8.1,'temp':30.2,'RH':24,'wind':2.7,'rain':0.0},
]
for i, row in enumerate(test_rows):
    result = predict_tabular(row)
    risk = "HIGH" if result['fire_probability'] > 0.6 else ("MEDIUM" if result['fire_probability'] > 0.35 else "LOW")
    print(f"  Sample {i+1}: Fire={result['fire_predicted']} | P(fire)={result['fire_probability']:.4f} | "
          f"Area={result['predicted_area_ha']:.2f}ha | Risk={risk}")


In [ ]:
# ── Test CSV input ────────────────────────────────────────────────────
import csv, io

csv_content = """X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain
7,5,aug,fri,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0
4,4,feb,mon,73.2,14.0,25.6,2.0,4.5,80,1.3,0.0
8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0
6,5,mar,sat,91.7,35.8,80.8,7.8,15.1,27,5.4,0.0
"""
df_test = pd.read_csv(io.StringIO(csv_content))
print("\nCSV Input Test:")
print(f"  Input rows: {len(df_test)}")

results_rows = []
for _, row in df_test.iterrows():
    r = predict_tabular(row.to_dict())
    row_dict = row.to_dict()
    row_dict.update(r)
    results_rows.append(row_dict)

df_results = pd.DataFrame(results_rows)
print(df_results[['X','Y','month','day','fire_predicted','fire_probability','predicted_area_ha']].to_string(index=False))

output_csv = PRED_DIR / "test_predictions.csv"
df_results.to_csv(output_csv, index=False)
print(f"\nPredictions saved: {output_csv}")


In [ ]:
# ── Error handling tests ──────────────────────────────────────────────
print("\nError Handling Tests:")

# 1. Corrupt image
try:
    bad_img = Image.new('RGB', (10, 10), color=(0,0,0))
    r = predict_image(bad_img)
    print(f"  ✓ Tiny image (10x10): {r['prediction']} ({r['confidence']:.3f})")
except Exception as e:
    print(f"  ✓ Tiny image handled: {e}")

# 2. Invalid CSV row  
try:
    bad_row = {'month': 'xxx', 'day': 'yyy', 'FFMC': 'abc'}
    r = predict_tabular(bad_row)
    print(f"  × Unknown month/day not caught — need validation in dashboard")
except Exception as e:
    print(f"  ✓ Invalid row caught: {type(e).__name__}")

# 3. Missing features → use 0
row_missing = {'X': 7, 'Y': 5, 'month': 'aug', 'day': 'fri', 'temp': 22}
r = predict_tabular(row_missing)
print(f"  ✓ Missing features (filled with 0): P(fire)={r['fire_probability']:.4f}")

print("\n" + "="*60)
print("ALL INFERENCE TESTS COMPLETE")
print("="*60)
print(f"  Image model: {MODEL_NAME} — READY")
print(f"  Classifier:  {tmeta['classifier_name']} — READY")
print(f"  Regressor:   {tmeta['regressor_name']} — READY")
print("  Dashboard can now be launched.")
